# Create a new bucket (resource version)

* [AWS resources](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/resources.html)
* [S3 resources](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3.html#resources)

https://boto3.amazonaws.com/v1/documentation/api/latest/guide/s3-example-creating-buckets.html

In [25]:
import sys
from dotenv import load_dotenv
import os

load_dotenv(override=True)

if "../app/" not in sys.path:
    sys.path.append("../app/")

print(sys.path)
# print(os.getenv("OPENAI_API_KEY"))

['/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/home/ryanwtsai/repos/lms-ai-agent/notebooks/venv/lib/python3.12/site-packages', '/home/ryanwtsai/repos/lms-ai-agent/notebooks/venv/lib/python3.12/site-packages/setuptools/_vendor', '/tmp/tmp87f2o4ar', '../app/']


In [26]:
import boto3
from boto3.session import Session
from dotenv import load_dotenv
import os, sys

load_dotenv(override=True)

if "../app/" not in sys.path:
    sys.path.append("../app/")

print(sys.path)
# print(os.getenv("OPENAI_API_KEY"))

bucket_name = "content-tagging-lms-lambda"

['/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/home/ryanwtsai/repos/lms-ai-agent/notebooks/venv/lib/python3.12/site-packages', '/home/ryanwtsai/repos/lms-ai-agent/notebooks/venv/lib/python3.12/site-packages/setuptools/_vendor', '/tmp/tmp87f2o4ar', '../app/']


In [27]:
# Create a session to override your default credentials (and the default session) with Beam Data credentials

session = Session(
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_REGION"),
)

In [28]:
# Create an S3 resource: https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3/service-resource/index.html

s3_resource = session.resource("s3")
print(type(s3_resource))

<class 'boto3.resources.factory.s3.ServiceResource'>


In [ ]:
# Create a bucket sub-resource: https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3/bucket/index.html

bucket = s3_resource.create_bucket(Bucket=bucket_name)
bucket.wait_until_exists()

In [ ]:
# Create a collection of Bucket resources: https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3/service-resource/buckets.html
# What is a collection? https://boto3.amazonaws.com/v1/documentation/api/latest/guide/collections.html

buckets = s3_resource.buckets
print(type(buckets))

# Check that your new bucket exists
for b in buckets.all():
    print(b)
    # print(b.name)

<class 'boto3.resources.collection.s3.bucketsCollectionManager'>
s3.Bucket(name='aws-logs-833659032354-us-east-1')
s3.Bucket(name='content-tagging-lms')
s3.Bucket(name='content-tagging-lms-lambda')
s3.Bucket(name='do-not-delete-ssm-diagnosis-833659032354-ca-central-1-jd2k2')
s3.Bucket(name='indeed-scrape')
s3.Bucket(name='jade-stack')
s3.Bucket(name='jade-youtube')
s3.Bucket(name='lms-analytics-2')
s3.Bucket(name='lms-vimeo-transcripts')
s3.Bucket(name='project-jade-youtube')
s3.Bucket(name='sql-haystack')
s3.Bucket(name='test-bucket-d2024')
s3.Bucket(name='test-jade-transcription')


## Upload a file to the bucket

In [44]:
with open("sample.ipynb", "w") as f:
    f.write("Sample bucket item!")

In [45]:
bucket.upload_file("./sample.ipynb", "sample.ipynb")

## Download a file from the bucket

In [31]:
bucket_name = "content-tagging-lms-lambda"
bucket = s3_resource.Bucket(bucket_name)
bucket

s3.Bucket(name='content-tagging-lms-lambda')

In [32]:
from pathlib import Path

key = "sample.txt"

file_name = Path(f"./s3_files/{key}")
file_name.parent.mkdir(exist_ok=True)
try:
    response = bucket.download_file(key, file_name)
except Exception as e:
    print(e)
    print(f"Error getting object {key} from bucket {bucket.name}. Make sure they exist and your bucket is in the same region as this function.")
    raise(e)

## Create a bucket notification

In [ ]:
# https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3/bucketnotification/index.html

bucket_notification = s3_resource.BucketNotification(bucket_name)
print(type(bucket_notification))

<class 'boto3.resources.factory.s3.BucketNotification'>


In [8]:
bucket_notification.lambda_function_configurations

In [ ]:
# https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3/bucketnotification/put.html

# Get bucket objects

In [49]:
bucket_name = "content-tagging-lms"
bucket = s3_resource.Bucket(bucket_name)
print(bucket.bucket_region)
print(bucket.creation_date)

None
2025-02-18 22:39:12+00:00


In [51]:
for o in bucket.objects.limit(10):
    print(o.key)

4350_vimeo_videos.parquet
4350_vimeo_videos_cleaned.parquet
Content/AWS introduction/AWS VPC & Networking.pptx
Content/AWS introduction/Chapter_Intro.md
Content/AWS introduction/Deploy Lambda Function from S3.md
Content/AWS introduction/Installing DBT on Ubuntu EC2.md
Content/AWS introduction/Installing H2O on Ubuntu EC2.md
Content/AWS introduction/Introduction to AWS.pptx
Content/AWS introduction/Introduction to Cloud.pptx
Content/AWS introduction/Introduction to EC2.pptx


In [15]:
o = s3_resource.Object(bucket_name="content-tagging-lms", key='Content/AWS introduction/Deploy Lambda Function from S3.md')
print(type(o))

<class 'boto3.resources.factory.s3.Object'>


In [57]:
from pathlib import Path

file_path = Path(o.key)
# file_path = Path("sample.txt")
print(file_path.parent)
print(file_path.name)

Content/AWS introduction
Introduction to EC2.pptx


# Create the lambda function

In [ ]:
from pymilvus import MilvusClient
import os

client = MilvusClient(
    uri=os.environ.get("ZILLIZ_CLUSTER_ENDPOINT"),
    token=os.environ.get("ZILLIZ_CLUSTER_TOKEN"),
)

In [9]:
client.list_collections()

['ryan_test_collection']

In [12]:
from pymilvus import MilvusClient, DataType

embed_dim = 768
# embed_dim = 128
max_content_len_chars = 65535
collection_name = "ryan_test_collection"

schema = MilvusClient.create_schema(
    auto_id=False,
    enable_dynamic_field=True,
)

schema.add_field(field_name="id", datatype=DataType.VARCHAR, is_primary=True, auto_id=False, max_length=64) # SHA-256
schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=embed_dim)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=max_content_len_chars)
schema.add_field(field_name="metadata", datatype=DataType.JSON)

index_params = client.prepare_index_params()

index_params.add_index(
    field_name="vector",
    metric_type="COSINE", # sentence-transformers/all-mpnet-base-v2 embeddings are L2-normalized
    index_type="AUTOINDEX",
    index_name="vector",
)

client.create_collection(
    collection_name=collection_name,
    schema=schema,
    index_params=index_params,
    consistency_level="Strong",
)

In [ ]:
data = [
    {"id": "1", "text": "dummy", "vector": [0]*768, "metadata": {}}
]

client.insert(
    collection_name=collection_name,
    data=data,
)

2025-04-10 08:25:36,811 [ERROR][handler]: RPC error: [insert_rows], <DataNotMatchException: (code=1, message=Insert missed an field `metadata` to collection without set nullable==true or set default_value)>, <Time:{'RPC start': '2025-04-10 08:25:36.705926', 'RPC error': '2025-04-10 08:25:36.811082'}> (decorators.py:140)


DataNotMatchException: <DataNotMatchException: (code=1, message=Insert missed an field `metadata` to collection without set nullable==true or set default_value)>

In [ ]:
# Original code: https://docs.aws.amazon.com/lambda/latest/dg/with-s3-example.html

import os
import json
import urllib.parse
import uuid
import boto3
from pathlib import Path
from haystack.components.routers import FileTypeRouter
from haystack_utilities.pipelines import indexing_pipeline_lambda
import haystack_utilities.tools

print("Loading environment variables.")

if not(os.environ.get("OPENAI_API_KEY")):
    raise Exception("Please provide an OPENAI_API_KEY environment variable.")

if not(os.environ.get("ZILLIZ_CLUSTER_ENDPOINT")):
    raise Exception("Please provide a ZILLIZ_CLUSTER_ENDPOINT environment variable (uri of your Zilliz cluster).")

if not(os.environ.get("ZILLIZ_CLUSTER_TOKEN")):
    raise Exception("Please provide a ZILLIZ_CLUSTER_TOKEN environment variable.")

if not(collection_name := os.environ.get("COLLECTION_NAME")):
    raise Exception("Please provide a str COLLECTION_NAME environment variable. This is the name of your Zilliz collection.")

splitting_options = os.environ.get("SPLITTING_OPTIONS", None)
if not splitting_options:
    # sentence-transformers/all-mpnet-base-v2
    # Max tokens = 384
    # Roughly 288 words (3 words = 4 tokens)
    splitting_options = {
        "split_strategy": "fixed",
        "split_by": "word",
        "split_length": 250,
        "split_overlap": 50,
        "split_threshold": 30,
        "respect_sentence_boundary": True
    }
else:
    try:
        splitting_options = json.loads(splitting_options)
    except json.JSONDecodeError as e:
        raise Exception("Please provide SPLITTING_OPTIONS environment variable in valid JSON format. This defines the chunking strategy.")

skip_cleaner = os.environ.get("SKIP_CLEANER", "True").lower() == "true"
add_tagger = os.environ.get("ADD_TAGGER", "True").lower() == "true"
max_content_len_chars = int(os.environ.get("MAX_CHUNK_LENGTH_IN_CHARS", 65535))

print("Building indexing pipeline.")
drop_old = True # If collection doesn't exist, create a new collection
with haystack_utilities.tools.MilvusContextManager() as client:
    if collection_name in client.list_collections():
        drop_old = False # If collection exists, do not create a new collection
index_pipe = indexing_pipeline_lambda.build_indexing_pipeline(collection_name, 
                                                              splitting_options, 
                                                              skip_cleaner=skip_cleaner, 
                                                              add_tagger=add_tagger, 
                                                              max_content_len_chars=max_content_len_chars, 
                                                              drop_old=drop_old)

print("Initializing file type router.") # Early termination if unsupported file type
file_type_router = FileTypeRouter(mime_types=haystack_utilities.tools.mime_types, additional_mimetypes=haystack_utilities.tools.additional_mimetypes)

print("Initializing AWS S3 connection.")
s3 = boto3.resource("s3")

print("Loading function.")
def lambda_handler(event, context):
    print("Received event: " + json.dumps(event, indent=2))

    # For now, the function is only expected to process one file at a time (typical S3 -> Lambda pipeline), so there is only one Record
    # Update later if batch processing is desired
    bucket_name = event['Records'][0]['s3']['bucket']['name']
    key = urllib.parse.unquote_plus(event['Records'][0]['s3']['object']['key'], encoding='utf-8')
    
    # Check that key is a valid file type
    if "unclassified" in file_type_router.run(sources=[Path(key)]).keys():
        raise Exception(f"{key} is not a supported MIME type. Supported MIME types are {haystack_utilities.tools.mime_types}")
    
    # Download the file
    bucket = s3.Bucket(bucket_name)
    try:
        file_name = Path(f"/tmp/{uuid.uuid4()}/{Path(key).name}")
        file_name.parent.mkdir()
        bucket.download_file(key, file_name)
    except Exception as e:
        print(e)
        print(f"Error getting object {key} from bucket {bucket_name}. Make sure they exist and your bucket is in the same region as this function.")
        raise e
    
    # Run indexing pipeline
    try:
        index_results = index_pipe.run({"file_type_router": {"sources": [file_name]}})
        print(f"Completed indexing job for {key}. {index_results["writer"]["documents_written"]} documents written to Zilliz cluster.")
    except Exception as e:
        print(e)
        print(f"Error when running the indexing pipeline.")
        raise e
    
    # Delete the file
    try:
        file_name.unlink()
        file_name.parent.rmdir()
    except Exception as e:
        print(e)
        print(f"Error deleting directory and file {file_name.as_posix()}")

    return index_results

In [19]:
import uuid
file_name = Path(f"./tmp/{uuid.uuid4()}/sample.txt")
file_name.parent.mkdir(parents=True)
# file_name.touch()
# Path("./pathdir/sample.txt").touch()

In [20]:
file_name.touch()

In [21]:
file_name.unlink()

In [23]:
file_name.parent.rmdir()

FileNotFoundError: [Errno 2] No such file or directory: 'tmp/dc2cd241-d6c3-44b7-a2a1-befcf23ca6d6'

# Create a new bucket (client version)

In [17]:
region_name = os.getenv("AWS_REGION")
s3_client = boto3.client(
    "s3",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=region_name,
)

print(type(s3_client))

<class 'botocore.client.S3'>


In [20]:
# List buckets

for b in s3_client.list_buckets()["Buckets"]:
    print(b["Name"])

aws-logs-833659032354-us-east-1
content-tagging-lms
content-tagging-lms-lambda
do-not-delete-ssm-diagnosis-833659032354-ca-central-1-jd2k2
indeed-scrape
jade-stack
jade-youtube
lms-analytics-2
lms-vimeo-transcripts
project-jade-youtube
sql-haystack
test-bucket-d2024
test-jade-transcription


In [14]:
import re

bucket_name = "content-tagging-lms"
pattern = re.compile(r"\bLLM\b")
pattern = re.compile(r".pdf")
for o in s3_client.list_objects(Bucket=bucket_name)["Contents"]:
    if pattern.search(o["Key"]):
        print(o["Key"])

Content/Excel Fundamental (self-paced)/Excel.pdf
Content/Excel Fundamental (self-paced)/FinTech/Intro to FinTech.pdf
Content/Excel Fundamental (self-paced)/Introduction to Excel.pdf


In [1]:
# How to create notification?

import logging
import boto3
from botocore.exceptions import ClientError

# try:
#     # s3_client.create_bucket(Bucket="content-tagging-lms-lambda",
#     #                         CreateBucketConfiguration={"LocationConstraint": region_name})
#     s3_client.create_bucket(Bucket="content-tagging-lms-lambda")
# except ClientError as e:
#     logging.error(e)

# for b in s3_client.list_buckets()["Buckets"]:
#     print(b["Name"])

In [8]:
s3 = boto3.resource("s3")

In [ ]:
s3.BucketNotification

s3.ServiceResource()

In [24]:
type(s3)

boto3.resources.factory.s3.ServiceResource

In [25]:
dir(s3)

['Bucket',
 'BucketAcl',
 'BucketCors',
 'BucketLifecycle',
 'BucketLifecycleConfiguration',
 'BucketLogging',
 'BucketNotification',
 'BucketPolicy',
 'BucketRequestPayment',
 'BucketTagging',
 'BucketVersioning',
 'BucketWebsite',
 'MultipartUpload',
 'MultipartUploadPart',
 'Object',
 'ObjectAcl',
 'ObjectSummary',
 'ObjectVersion',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'buckets',
 'create_bucket',
 'get_available_subresources',
 'meta']

In [9]:
for b in s3.buckets.all():
    print(b)

s3.Bucket(name='ryan-demo-bucket22cd1de61e40491a88da48bcae16fdef')
s3.Bucket(name='ryan-end-to-end')


In [31]:
bucket = s3.Bucket(name='content-tagging-lms-lambda')

In [32]:
bucket.delete()

{'ResponseMetadata': {'RequestId': 'CX9MY0BANSZEAQDE',
  'HostId': 'HqBcWI/07cuXNpZkrGqfpYLrjx69Uql8gpBBDKK68XEm3aFyZzksNj3yrJpO2JHFvnuLsSRUCYo=',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'x-amz-id-2': 'HqBcWI/07cuXNpZkrGqfpYLrjx69Uql8gpBBDKK68XEm3aFyZzksNj3yrJpO2JHFvnuLsSRUCYo=',
   'x-amz-request-id': 'CX9MY0BANSZEAQDE',
   'date': 'Fri, 04 Apr 2025 23:18:16 GMT',
   'server': 'AmazonS3'},
  'RetryAttempts': 1}}

In [33]:
bucket.wait_until_not_exists()

In [35]:
bucket.bucket_region